In [ ]:
!pip install -q arize-otel langchain langchain-openai openinference-instrumentation-langchain python-dotenv

In [ ]:
from arize.otel import register, Transport
from dotenv import load_dotenv
import os

load_dotenv()

# HOST = os.getenv("ARIZE_HOST")
API_KEY = os.getenv("ARIZE_API_KEY")
SPACE_ID = os.getenv("ARIZE_SPACE_ID")  

tracer_provider = register(
    # endpoint=f"{HOST}/v1/traces",
    # transport=Transport.HTTP,
    space_id = SPACE_ID,
    api_key = API_KEY,
)

# Import the automatic instrumentor from OpenInference
from openinference.instrumentation.langchain import LangChainInstrumentor

# Finish automatic instrumentation
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

In [ ]:
# Agent 구성 및 실행
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def get_weather(location: str):
    """Call to get the weather from a specific location."""
    # This is a placeholder for the actual implementation
    # Don't let the LLM know this though 😊
    if any([city in location.lower() for city in ["sf", "san francisco"]]):
        return "It's sunny in San Francisco, but you better look out if you're a Gemini 😈."
    else:
        return f"I am not sure what the weather is in {location}"

# 날씨 조회 Tool 추가
tools = [get_weather]

# 날씨 조회 Agent 구성
agent = create_agent(
    model="gpt-4o",
    tools=tools,
    system_prompt="You are a helpful assistant.",
)

# 날씨 조회 Agent 실행
agent.invoke({"input": "hi, what is the weather in sf?", "user_id": "user1"})